In [1]:
import sys
import os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import torch
from accelerate.test_utils.testing import get_backend
from core.models import LinearRegression
from core.callbacks import WandBCallback
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
from core.data import FullBatchDataModule
from core.estimators import BiasWithMSE
from core.utils import compute_Q_matrix, compute_beta_closed_form
from lightning.pytorch.callbacks import EarlyStopping
device, n_devices, _ = get_backend()
# device = "cpu"
# n_devices = 1

torch.set_float32_matmul_precision("highest")

## Ali et al. Linear Regression

In [2]:
# set seed ensure reproducibility
torch.manual_seed(56)
# deterministic behavior
torch.use_deterministic_algorithms(False)

p = 5 # input dimension / number of features
N = 1000 # number of samples
X = torch.randn(N, p)
# X = torch.diag(torch.randn(p))
betas = 3 * torch.randn(p)
y = X @ betas
dm_linear = FullBatchDataModule(X, y, num_workers=3)

In [3]:
eps = 1e-3
linear_model = LinearRegression(input_dim=p, 
                                output_dim=1, 
                                lr=eps/2, # fix mismatched learning rate due to 1/2 factor in the loss (not in torch)
                                fit_intercept=False, init_zeros=True)
wandb_logger = WandbLogger(
    project="inductive-bias", name="linear-regression", log_model=False
)

lm_trainer = Trainer(
    max_epochs=500,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=5, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
lm_trainer.fit(linear_model, dm_linear)

/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /shared_data0/jrudoler/.cache/pypoetry/virtualenvs/i ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type    | Params | Mode 
----------------------------------------------
0 | linear    | Linear  | 5      | train
1 | loss_func | MSELoss | 0      | train
----------------------------------------------
5         Trainable params
0         Non-trainable params
5         Total params
0.000     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=500` reached.


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇██████
train/loss,███▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█
epoch,499
train/loss,11.5037
trainer/global_step,499


In [4]:
k = lm_trainer.current_epoch
# Compute beta iterates up to k
beta_iterates = [torch.zeros(p)]  # Initialize with beta_0
for i in range(1, k + 1):
    beta_prev = beta_iterates[-1]
    # beta_next = beta_prev + 2 * eps * (X.T @ y - X.T @ X @ beta_prev) / N # depends on the learning rate
    beta_next = beta_prev + eps * (X.T @ y - X.T @ X @ beta_prev) / N
    beta_iterates.append(beta_next)
    # print(f"Beta at step {i}: {beta_iterates[-1]}")
    
model_beta = linear_model.linear.weight.detach()
print(f"Final beta iterate: {beta_iterates[-1]}")
print(f"Model beta: {model_beta}")
assert torch.allclose(beta_iterates[-1], model_beta, atol=1e-2), "Final beta does not match the model's weight"

Final beta iterate: tensor([ 0.2815, -0.5451,  1.2348,  1.6046,  0.6029])
Model beta: tensor([[ 0.2815, -0.5451,  1.2348,  1.6046,  0.6029]])


In [5]:
print(list(linear_model.parameters()))
print(betas)
assert torch.allclose(betas, linear_model.linear.weight[0], atol=1e-1)


[Parameter containing:
tensor([[ 0.2815, -0.5451,  1.2348,  1.6046,  0.6029]], requires_grad=True)]
tensor([ 0.5684, -1.4530,  3.1867,  3.9853,  1.6735])


AssertionError: 

In [6]:
k = lm_trainer.current_epoch
print(f"epoch: {k}")

Q = compute_Q_matrix(X, k, eps)
print(Q)

epoch: 500
tensor([[ 1.5608e+00,  6.9450e-03, -2.1399e-02, -5.1016e-04, -6.1263e-03],
        [ 6.9449e-03,  1.5570e+00, -1.5629e-02,  1.5743e-02, -1.7101e-02],
        [-2.1399e-02, -1.5629e-02,  1.5135e+00,  2.0703e-02,  1.1517e-02],
        [-5.1017e-04,  1.5743e-02,  2.0703e-02,  1.5233e+00, -1.4522e-02],
        [-6.1263e-03, -1.7101e-02,  1.1517e-02, -1.4522e-02,  1.5851e+00]])


In [7]:
beta_closed_form = compute_beta_closed_form(X, y, Q)
# Compare with model's beta
print(f"Closed-form solution: {str(beta_closed_form.detach().numpy())}")
print(f"Model beta: {str(model_beta[0].detach().numpy())}")

Closed-form solution: [ 0.28158757 -0.54505694  1.2348183   1.6046878   0.6029629 ]
Model beta: [ 0.28151843 -0.5450531   1.234768    1.6046127   0.602941  ]


In [12]:
from core.estimators import BiasWithMSE, BiasWithAutodiffLoss
from core.bias import MatrixRidgeBias, DiagMatrixRidgeBias, RidgeBias

torch.manual_seed(56)
# noisy_Q_init = Q.clone() + 0.1 * torch.randn(Q.shape[0], Q.shape[1])
# matrix_bias_model = MatrixRidgeBias(dim=p, Q_init=noisy_Q_init.clone())

matrix_bias_model = DiagMatrixRidgeBias(dim=p)
# matrix_bias_model = RidgeBias()
wandb_logger = WandbLogger(
    project="inductive-bias", name="matrix-ridge-bias", log_model=False
)

estimator = BiasWithMSE(
    predictive_model=linear_model,
    # predictive_loss_fn=torch.nn.functional.mse_loss,
    bias_model=matrix_bias_model,
    grad_match_loss_fn=torch.nn.functional.mse_loss,
    lr=1e-2,
    optimizer_cls=torch.optim.Adam,
)

trainer = Trainer(
    max_epochs=5000,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=150, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
trainer.fit(estimator, dm_linear)

# Q_hat = torch.eye(p) * estimator.bias_model.beta.detach()
Q_hat = torch.diag(estimator.bias_model.Q.detach())

# minibatch_dataloader = torch.utils.data.DataLoader(TensorDataset(X, y), batch_size=N//2, shuffle=True)
# trainer.fit(estimator, train_dataloaders=minibatch_dataloader)

/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /shared_data0/jrudoler/.cache/pypoetry/virtualenvs/i ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                | Params | Mode 
-----------------------------------------------------------------
0 | predictive_model | LinearRegression    | 5      | train
1 | bias_model       | DiagMatrixRidgeBias | 5      | train
-----------------------------------------------------------------
10        Trainable params
0         Non-trainable params
10        Total params
0.000     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

bias/Q,▁▂▂▂▃▅▆▆▇▇▇▇▇███████████████████████████
bias/Q_grad,██▇▇▇▆▆▆▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇███
train/loss,█▅▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇█████
bias/Q,3.42696
bias/Q_grad,1e-05
epoch,839
train/loss,0.0
trainer/global_step,839


In [13]:
evals = torch.linalg.eigvalsh(Q)
print(f"Eigenvalues of Q: {evals}")
print(f"max eigenvalue: {evals.max()}")
print(f"min eigenvalue: {evals.min()}")

Eigenvalues of Q: tensor([1.4860, 1.5317, 1.5501, 1.5682, 1.6037])
max eigenvalue: 1.6036514043807983
min eigenvalue: 1.486008644104004


In [14]:
# print("initial:\t", torch.diag(noisy_Q_init))
print("estimated:\n", Q_hat)
print("theoretical:\n", Q)

estimated:
 tensor([[1.4381, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 1.5614, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.5481, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.5285, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.5827]])
theoretical:
 tensor([[ 1.5608e+00,  6.9450e-03, -2.1399e-02, -5.1016e-04, -6.1263e-03],
        [ 6.9449e-03,  1.5570e+00, -1.5629e-02,  1.5743e-02, -1.7101e-02],
        [-2.1399e-02, -1.5629e-02,  1.5135e+00,  2.0703e-02,  1.1517e-02],
        [-5.1017e-04,  1.5743e-02,  2.0703e-02,  1.5233e+00, -1.4522e-02],
        [-6.1263e-03, -1.7101e-02,  1.1517e-02, -1.4522e-02,  1.5851e+00]])


In [15]:
print("estimated:", compute_beta_closed_form(X, y, Q_hat))
print("theoretical:", compute_beta_closed_form(X, y, Q))
print("ground truth:", linear_model.linear.weight[0].detach().numpy().round(5))

estimated: tensor([ 0.2815, -0.5451,  1.2348,  1.6046,  0.6029])
theoretical: tensor([ 0.2816, -0.5451,  1.2348,  1.6047,  0.6030])
ground truth: [ 0.28152 -0.54505  1.23477  1.60461  0.60294]


## How good are heuristics?

If we only fit the scalar $\lambda$ penalty term (which we know is not fully descriptive of the bias), does it still
give us a useful estimate of the bias?
i.e. theory tellls us $\lambda$ should decrease as k increases, is this true in practice?
 

In [ ]:
# set seed ensure reproducibility
torch.manual_seed(56)
# deterministic behavior
torch.use_deterministic_algorithms(False)

p = 5 # input dimension / number of features
N = 1000 # number of samples
X = torch.randn(N, p)
# X = torch.diag(torch.randn(p))
betas = 3 * torch.randn(p)
y = X @ betas
dm_linear = FullBatchDataModule(X, y, num_workers=3)

eps = 1e-2
linear_model = LinearRegression(input_dim=p, 
                                output_dim=1, 
                                lr=eps/2, # fix mismatched learning rate due to 1/2 factor in the loss (not in torch)
                                fit_intercept=False, init_zeros=True)
wandb_logger = WandbLogger(
    project="inductive-bias", name="linear-regression", log_model=False
)

lm_trainer = Trainer(
    max_epochs=500,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=5, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
lm_trainer.fit(linear_model, dm_linear)

In [ ]:
from core.estimators import BiasWithMSE, BiasWithAutodiffLoss
from core.bias import MatrixRidgeBias, DiagMatrixRidgeBias, RidgeBias

torch.manual_seed(56)

bias_model = RidgeBias()
wandb_logger = WandbLogger(
    project="inductive-bias", name="matrix-ridge-bias", log_model=False
)

estimator = BiasWithAutodiffLoss(
    predictive_model=linear_model,
    predictive_loss_fn=torch.nn.functional.mse_loss,
    bias_model=bias_model,
    grad_match_loss_fn=torch.nn.functional.mse_loss,
    lr=1e-2,
    optimizer_cls=torch.optim.Adam,
)

trainer = Trainer(
    max_epochs=5000,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=150, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
trainer.fit(estimator, dm_linear)

/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /shared_data0/jrudoler/.cache/pypoetry/virtualenvs/i ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type             | Params | Mode 
--------------------------------------------------------------
0 | predictive_model | LinearRegression | 5      | train
1 | bias_model       | RidgeBias        | 1      | train
--------------------------------------------------------------
6         Trainable params
0         Non-trainable params
6         Total params
0.000     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,█▇▅▅▅▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▆▆▆▇▇▇█████
train/loss,█▇▇▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇██
bias/beta,1e-05
epoch,601
train/loss,0.0
trainer/global_step,601


In [12]:
bias_model.beta

Parameter containing:
tensor([6.8130e-06], requires_grad=True)

10 epochs: $\lambda = 0.00946$

50 epochs: $\lambda = 0.00153$

150 epochs: $\lambda = 0.00028$